# Cell typing – updated to match IHOPE project

# Prep

In [ ]:
import pandas as pd
from anndata import AnnData, read_h5ad
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc

Make sure you're in the parent directory:

In [ ]:
import sys
from pathlib import Path
#TODO try to remove this somehow
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

If you already have a H5AD file with an anndata object:

In [ ]:
from scripts.anndata_helpers import load_and_build_anndata, save_h5ad

basename = "IHOPE26_LN_InstanSeq_RAW_RESULTS"

# Read file and create AnnData object
filename = f"../data/processed/{basename}_cleaned_filtered_arcsinh_cf5.0.csv"
adata = load_and_build_anndata(filename)

# Preprocessing

**If it's the first time running the sample:**

Load, filter and normalize your raw data in csv format. Also, make sure to specify which file you're working with to facilitate later analysis. All subsequent naming of files is based on this.

In [ ]:
from scripts import preprocessing
import importlib

# Reload to ensure notebook uses the latest version of the script
importlib.reload(preprocessing)

# Input path, here you need to write the entire raw data file name
csv_in = "../data/raw/IHOPE14_MedLN_BottomLeft.csv"

# Keep track of which sample you're working on:
basename = "IHOPE14_MedLN_BottomLeft"

# Output path, you can use this default format based on your basename:
csv_out = f"../data/processed/{basename}_cleaned_raw.csv"

# Clean raw CSV
preprocessing.clean_cell_columns(csv_in, csv_out)

In [ ]:
from scripts import preprocessing
import importlib

# Reload to ensure the latest version of the script is used
importlib.reload(preprocessing)

# Run preprocessing
input_file = f"../data/processed/{basename}_cleaned_raw.csv"
output_file = f"../data/processed/{basename}_cleaned_filtered.csv"

preprocessing.preprocess(input_file, output_file, plot=True)


For really large files:

In [ ]:
from scripts import preprocessing
import importlib

importlib.reload(preprocessing)

# Marker intensity columns
TARGET_COLUMNS = [
    "CCR6: Mean", "CCR7: Mean", "CD107a: Mean", "CD11c: Mean", "CD14: Mean", "CD141: Mean",
    "CD163: Mean", "CD1c: Mean", "CD20: Mean", "CD21: Mean", "CD27: Mean", "CD31: Mean",
    "CD34: Mean", "CD38: Mean", "CD3e: Mean", "CD4: Mean", "CD40: Mean", "CD45: Mean",
    "CD45RA: Mean", "CD45RO: Mean", "CD57: Mean", "CD68: Mean", "CD69: Mean", "CD79a: Mean",
    "CD8: Mean", "Collagen IV: Mean", "CXCL13: Mean", "DAPI: Mean", "FOXP3: Mean",
    "Granzyme B: Mean", "HLA-DR: Mean", "ICOS: Mean", "IFNG: Mean", "Ki67: Mean", "LYVE1: Mean",
    "PD-1: Mean", "TCF-1: Mean", "Vimentin: Mean"
]

OBS_COLUMNS = [
    "Object ID",
    "Centroid X µm",
    "Centroid Y µm",
    "Area µm^2",
    "Nucleus/Cell area ratio"
]

csv_in = "../data/raw/IHOPE14_Spleen_RAW_RESULTS_combined.csv"

basename = "IHOPE14_Spleen"

csv_out = f"../data/processed/{basename}_cleaned_filtered.csv"

preprocessing.extract_and_filter_columns_chunked(
    csv_in,
    csv_out,
    TARGET_COLUMNS,
    OBS_COLUMNS
)



**Normalize**

Choose between: "arcsinh", "z-score" and "none".

For arcsinh, you can specify a cofactor.

In [ ]:
from scripts.transforms import apply_transform

method = "arcsinh"
cofactor = 5.0    # Comment out if not applying arcsinh
input_file = f"../data/processed/{basename}_cleaned_filtered.csv"
output_file = f"../data/processed/{basename}_cleaned_filtered_arcsinh_cf5.0.csv"

df_out, markers, metadata, fig = apply_transform(
    input_file=input_file,
    method=method,
    cofactor=cofactor, # Comment out if not applying arcsinh
    output_file=output_file,
    save_plot=True
)

# Building Anndata and annotating marker positivity


In [ ]:
import importlib
import scripts.anndata_helpers

# Reload the module after editing
importlib.reload(scripts.anndata_helpers)

In [ ]:
from scripts.anndata_helpers import load_and_build_anndata, save_h5ad

# Read file and create AnnData object
filename = f"../data/processed/{basename}_cleaned_filtered_arcsinh_cf5.0.csv"
adata = load_and_build_anndata(filename)

In [ ]:
print("adata summary: ")
print(adata)
print("adata X shape: ")
print(adata.X.shape)
print("adata spatial data dimensions: ")
print(adata.obsm['spatial'].shape)
print("adata var: ")
print(adata.var)
print("adata var names: ")
print(adata.var_names)

Apply marker positivity thresholds (GMM intersections):

In [ ]:
from scripts.annotation import compute_positivity_matrix

# Annotate all cells with GMM-based positivity for canonical markers, fallback to percentile if unimodal
adata, thresholds, best_gmms = compute_positivity_matrix(
    adata,
    quantile=0.8,
    random_state=0
)

In [ ]:
# Visualize the distributions and thresholds:
from scripts.annotation import plot_marker_gmm_adata
plot_marker_gmm_adata(adata,
                      thresholds,
                      best_gmms,
                      title_prefix=f"{basename} Marker: ",
                      save=False,
                      filename=f"{basename}_GMM_histograms")

In [ ]:
print(f"{basename}_GMM_thresholds")

Optional: add high/low levels to marker of interest.

Specify marker name, quantiles to label as low and high, and a recommended plot for visualization.

In [ ]:
import importlib
import scripts.annotation

# Reload the module after editing
importlib.reload(scripts.annotation)

In [ ]:
from scripts import annotation

annotation.add_intensity_tiers(
    adata,
    marker="CD38",
    low_q=0.33,
    high_q=0.9,
    plot=True
)

In [ ]:
from scripts import annotation

annotation.add_intensity_tiers(
    adata,
    marker="CD21",
    low_q=0.33,
    high_q=0.9,
    plot=True
)

Save anndata object with progress so far

In [ ]:
# Save AnnData as h5ad
h5adpath = f"../data/processed/anndata/{basename}_filtered_arcsinh_cf5_GMM.h5ad"
save_h5ad(adata, h5adpath)

# Rule-based cell typing

Each cell type will correspond to a column in the AnnData object, with a boolean True/False for every cell type in every cell.

If you want to clean up before you rerun cell type assignment:

In [ ]:
import scripts
import scripts.celltype_rules_IHOPE
import importlib

importlib.reload(scripts.celltype_rules_IHOPE)

In [ ]:
#If you need to load anndata:

import anndata as ad

basename = "IHOPE26_Spleen"
adata = ad.read_h5ad(
    f"../data/processed/anndata/{basename}_filtered_arcsinh_cf5_GMM.h5ad"
)

In [ ]:
from scripts.celltype_rules_IHOPE import assign_cell_types_bool_IHOPE

adata = assign_cell_types_bool_IHOPE(adata)


Save anndata with cell type information:

In [ ]:
h5adpath = f"../data/processed/anndata/{basename}_filtered_arcsinh_cf5_GMM_IHOPE_celltypes.h5ad"
save_h5ad(adata, h5adpath)

# Visual diagnostics

Some FACS-style plots to "diagnose" how the thresholds look:

In [ ]:
import scripts
import scripts.marker_plots
import importlib

importlib.reload(scripts.marker_plots)

In [ ]:
from scripts.marker_plots import plot_marker_axes

plot_marker_axes(
    adata,
    x_marker="CD79a",
    y_marker="CD20",
    base_mask="type_B",
    title=f"B cells: CD45 vs CD20",
    show_thresholds = True
)

In [ ]:
plot_marker_axes(
    adata,
    x_marker="CD45",
    y_marker="CD3e",
    base_mask="type_T",
    title="T cells: CD45 vs CD3e"
)

Marker spatial plots:

In [ ]:
import scripts
from scripts.spatialmapping import plot_spatial_marker

# One marker
plot_spatial_marker(adata, "CD20", thresholded=True, title_prefix="GMM thresholding: ", color_hi="red", color_lo="lightblue")

plot_spatial_marker(adata, "CD79a", thresholded=True, title_prefix="GMM thresholding: ", color_hi="red", color_lo="lightblue")


Heatmap of markers in cell types:

In [ ]:
import scripts
import scripts.plotting
import scripts.spatial_plot
import importlib

importlib.reload(scripts.plotting)
importlib.reload(scripts.spatial_plot)


In [ ]:
from scripts.plotting import clustered_marker_heatmap

# all cell type columns
celltype_cols = [
    c for c in adata.obs.columns
    if c.startswith(("type_", "state_", "subtype_"))
]

matrix = clustered_marker_heatmap(
    adata,
    celltype_cols=celltype_cols,
    min_cells=15,
    base_fig_width=10,
    fig_height=12,
    cmap="viridis",
)


# Spatial visualization

In [ ]:
import scripts
import scripts.spatial_plot
import importlib

importlib.reload(scripts.spatial_plot)

In [ ]:
from scripts.spatial_plot import spatial_celltype_plot

celltype_cols = [c for c in adata.obs.columns if c.startswith("type_") and not c.endswith("unclassified") and not c.endswith("Endothelial")]

#or use this for all:
#[c for c in adata.obs.columns if c.startswith(("type_", "state_", "subtype_"))]

# Spatial plot
spatial_celltype_plot(adata, celltype_cols, min_cells=50)


In [ ]:
print(f"{basename}_types_spatial")

In [ ]:
from scripts.spatial_plot import spatial_celltype_plot

# T cell subtype columns
t_subtypes = [c for c in adata.obs.columns if c.startswith("subtype_T")]

# Plot
spatial_celltype_plot(
    adata,
    celltype_cols=t_subtypes,
    min_cells=15,
    size=10,
    alpha=0.7
)

In [ ]:
print(f"{basename}_Tc_types_spatial")

In [ ]:
from scripts.spatial_plot import spatial_celltype_plot

# B cell subtype columns
b_subtypes = [c for c in adata.obs.columns if c.startswith("subtype_B_")]

# Plot
spatial_celltype_plot(
    adata,
    celltype_cols=b_subtypes,
    min_cells=15,
    size=10,
    alpha=0.7
)


In [ ]:
print(f"{basename}_Bc_types_spatial")

In [ ]:
import math
import matplotlib.pyplot as plt

celltypes = (
    [c for c in adata.obs.columns if c.startswith("type_")] +
    [c for c in adata.obs.columns if c.startswith("intermediate_")] +
    [c for c in adata.obs.columns if c.startswith("subtype_")]
)

# Filter by min_cells
celltypes = [c for c in celltypes if adata.obs[c].sum() >= 15]

n = len(celltypes)
cols = 4
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4))

axes = axes.flatten()

for ax, ct in zip(axes, celltypes):

    mask = adata.obs[ct].astype(bool)

    ax.scatter(
        adata.obs.loc[mask, "x"],
        adata.obs.loc[mask, "y"],
        s=8,
        alpha=0.7
    )

    ax.set_title(ct, fontsize=10)
    ax.invert_yaxis()
    ax.axis("off")

# remove empty panels
for ax in axes[n:]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
print(f"{basename}_celltypes_spatial_grid")


# Cell type summary in sample

Use after loading data and running annotation!

This will be used later for statistical comparison between tissues.

In [ ]:
import scripts
import scripts.summary_celltypes_IHOPE
import importlib

importlib.reload(scripts.summary_celltypes_IHOPE)

In [ ]:
from scripts.summary_celltypes_IHOPE import summarize_celltypes_IHOPE

df_summary = summarize_celltypes_IHOPE(
    adata,
    filename=f"{basename}_filtered_arcsinh_cf5.0_IHOPE_summary.csv"
)


# Comparison

Time to compare between samples!

In [383]:
import importlib
import scripts.comparison as comparison

importlib.reload(comparison)

from scripts.comparison import (
    load_celltype_summaries,
    pivot_for_heatmap,
    plot_celltype_heatmap,
    print_numeric_summary,
)


In [ ]:
#OOOOLD
df = load_celltype_summaries(
    summaries_dir=summaries_dir,
    basenames=basenames,
)

print_numeric_summary(df)

matrix = pivot_for_heatmap(df)

plot_celltype_heatmap(
    matrix,
    title="Cell type composition across samples (all levels)",
)


# BANKSY
For this part you need a saved .h5ad file containing an anndata object, and Python 12 is required (not a later version of Python). Note that BANKSY only identifies spatial domains and not cell types. This section provides the workflow from AnnData object to BANKSY domains, which can be used as a foundation for further analysis.

In [ ]:
basename = "IHOPE39_Spleen"

In [ ]:
from banksy.initialize_banksy import initialize_banksy
from banksy.run_banksy import run_banksy_multiparam
import scanpy as sc

h5ad_file = f"../data/processed/anndata/{basename}_filtered_arcsinh_cf5_GMM_IHOPE_celltypes.h5ad"

# Load your spatial transcriptomics data
adata = sc.read_h5ad(h5ad_file)

# Initialize BANKSY
coord_keys = ('x', 'y', 'spatial')
banksy_dict = initialize_banksy(
    adata,
    coord_keys=coord_keys,
    num_neighbours=15,
    nbr_weight_decay='scaled_gaussian'
)

# Run BANKSY clustering

results_df = run_banksy_multiparam(
    adata,
    banksy_dict,
    lambda_list=[0.2],
    resolutions=[0.5, 1.0]
)

In [ ]:
from banksy.main import median_dist_to_nearest_neighbour
from banksy.initialize_banksy import initialize_banksy
from banksy.embed_banksy import generate_banksy_matrix
from banksy_utils.umap_pca import pca_umap
from banksy.cluster_methods import run_Leiden_partition
from banksy.plot_banksy import plot_results

# Parameters for BANKSY
coord_keys = ('x', 'y', 'spatial')
k_geom = 15
max_m = 1
nbr_weight_decay = "scaled_gaussian"
lambda_list = [0.8]
resolutions = [0.5]       # Leiden clustering resolution
pca_dims = [20]             # Try a lower number?
cluster_algorithm = 'leiden'
cmap = 'tab20'             # color map for spatial plotting
save_path = None           # e.g., "./BANKSY_results" if you want to save figures

# Compute neighbor distances
nbrs = median_dist_to_nearest_neighbour(adata, key=coord_keys[2])

# Initialize BANKSY
banksy_dict = initialize_banksy(
    adata,
    coord_keys,
    k_geom,
    nbr_weight_decay=nbr_weight_decay,
    max_m=max_m,
    plt_edge_hist=False,
    plt_nbr_weights=True,
    plt_agf_angles=False,
    plt_theta=False
)

# Generate BANKSY matrix
banksy_dict, banksy_matrix = generate_banksy_matrix(
    adata,
    banksy_dict,
    lambda_list,
    max_m
)

# Dimensionality reduction by UMAP
pca_umap(
    banksy_dict,
    pca_dims=pca_dims,
    add_umap=True
)

# Run Leiden clustering
results_df, max_num_labels = run_Leiden_partition(
    banksy_dict,
    resolutions=resolutions,
    num_nn=50,
    num_iterations=-1,
    partition_seed=1234,
    match_labels=True,
    max_labels=None
)

# Map clusters back to adata
cluster_labels = results_df.labels[results_df.index[0]].dense
adata.obs['banksy_domain'] = cluster_labels.astype(str)

# Optional: visualize
sc.pl.spatial(adata, color='banksy_domain', spot_size=30, title='BANKSY Domains')

# Optional: use BANKSY plotting function
if save_path is not None:
    os.makedirs(save_path, exist_ok=True)
    weights_graph = banksy_dict['scaled_gaussian']['weights'][1]
    plot_results(
        results_df[results_df['num_labels']==len(np.unique(cluster_labels))],
        weights_graph,
        cmap,
        match_labels=True,
        coord_keys=coord_keys,
        max_num_labels=max_num_labels,
        save_path=save_path,
        save_fig=True,
        save_fullfig=True,
        dataset_name='Sample',
        save_labels=True
    )

print(f"BANKSY complete for {basename}! Domains added to `adata.obs['banksy_domain']`")

Save anndata object with BANKSY domains

In [ ]:
from scripts.anndata_helpers import load_and_build_anndata, save_h5ad

h5adpath = f"../data/processed/anndata/{basename}_filtered_arcsinh_cf5_GMM_IHOPE_celltypes_banksy.h5ad"
save_h5ad(adata, h5adpath)

In [ ]:
print(adata)

In [ ]:
# Deciphering cluster identity!

cluster_key = "banksy_domain"

# Clusterwise mean intensity heatmap:
#Mean expression per domain
cluster_means = (
    pd.DataFrame(
        adata.X,
        index=adata.obs_names,
        columns=adata.var_names
    )
    .groupby(adata.obs[cluster_key])
    .mean()
)

# Z-score across clusters
cluster_means_z = cluster_means.sub(cluster_means.mean(), axis=1)
cluster_means_z = cluster_means_z.div(cluster_means.std(), axis=1)

#Heatmap z-scored
sns.clustermap(
    cluster_means_z,
    cmap="vlag",
    center=0,
    figsize=(10, 12),
    col_cluster=True,
    row_cluster=True
)
plt.suptitle(f"{basename} BANKSY domains marker enrichment z-scored", y=1.02)
plt.show()

# Rows = BANKSY domains
# Columns = markers
# Red = enriched, Blue = depleted
# GC domains should light up for CD20, CD21, CD38, CXCL13
# T zones for CD3e, CD4, CD8
# Sinus / stroma for LYVE1, Vimentin, Collagen IV

# Step 2: differential analysis
# Rank markers per BANKSY domain –nonparametric test to compare markers in domain vs all other cells
# logFC = log( mean(M in D) / mean(M outside D) )
# Also the p-values are FDR-adjusted by default.
sc.tl.rank_genes_groups(
    adata,
    groupby=cluster_key,
    method="wilcoxon"
)

sc.pl.rank_genes_groups_dotplot(
    adata,
    n_genes=5,
    groupby=cluster_key,
    standard_scale="var"
)

# Dot size = fraction of cells in domain expressing marker (but all are above 0, hence same size)
# Color = average intensity

# B cell follicle domain annotation

Change file name/path and basename! 

In [384]:
# Load AnnData
adata = sc.read_h5ad(
    "../data/processed/anndata/IHOPE14_MedLN_BottomLeft_filtered_arcsinh_cf5_GMM_IHOPE_celltypes_banksy.h5ad"
)
basename = "IHOPE14_MedLN_BottomLeft"
print(f"Base name: {basename}")

Base name: IHOPE14_MedLN_BottomLeft


Clean up time and assigning cell types again

In [385]:
cols_to_drop = [
    c for c in adata.obs.columns
    if c.startswith(("type_", "intermediate_", "subtype_", "state_"))
]

adata.obs.drop(columns=cols_to_drop, inplace=True)

print(f"Removed {len(cols_to_drop)} old annotation columns")

Removed 26 old annotation columns


In [ ]:
from scripts.celltype_rules_IHOPE import assign_cell_types_bool_IHOPE

adata = assign_cell_types_bool_IHOPE(adata)

In [ ]:
from scripts.anndata_helpers import save_h5ad #load_and_build_anndata??

celltype_path = f"../data/processed/anndata/NEW/{basename}_celltypes.h5ad"

save_h5ad(adata, celltype_path)

Banksy annotation

In [ ]:
import importlib
import scripts.banksy_domains

# Reload the module after editing
importlib.reload(scripts.banksy_domains)

In [ ]:
from scripts.banksy_domains import compute_domain_bcell_stats, plot_domains_by_bcell_fraction

stats_df = compute_domain_bcell_stats(adata)
plot_domains_by_bcell_fraction(adata, stats_df, cmap="coolwarm")

In [ ]:
print(f"{basename}_BANKSY_B_cell_percentage_spatial")

Manually select top domains and add to anndata

In [ ]:
top_domains = ['8']  # <-- you set this per sample, within ''

In [ ]:
from scripts.banksy_domains import assign_bcell_follicles

adata = assign_bcell_follicles(
    adata,
    follicle_domains=top_domains,
    banksy_domain_key="banksy_domain",
    output_key="B_follicle",
)

print("Sanity check sum:", adata.obs["B_follicle"].sum())

Quick check of the follicle domain:

In [ ]:
from scripts.banksy_domains import plot_domain_mask
plot_domain_mask(adata, top_domains)

Plot of the B cells within the selected domain

In [ ]:
from scripts.banksy_domains import plot_bcell_follicles

plot_bcell_follicles(adata, sample_name = basename)

In [ ]:
print(f"{basename}_BANKSY_follicles_domain")

In [ ]:
follicle_path = f"../data/processed/anndata/NEW/{basename}_celltypes_follicledomains.h5ad"

save_h5ad(adata, follicle_path)

Add spatially defined cell types:

In [ ]:
# TFH-LIKE
from scripts.celltype_rules_IHOPE import add_TfH_like_cells

adata = add_TfH_like_cells(
    adata,
    follicle_key="B_follicle",
    plot=False,
    size=0.3,
    sample_name=basename
)

In [ ]:
#GC-B AND PLASMABLASTS
from scripts.celltype_rules_IHOPE import add_spatial_B_context

adata = add_spatial_B_context(
    adata,
    follicle_key="B_follicle",
    plot=False,
    size=0.3,
    sample_name=basename
)

In [ ]:
final_path = f"../data/processed/anndata/NEW/{basename}_celltypes_follicledomains.h5ad"

save_h5ad(adata, final_path)

# Summarize

In [ ]:
import scripts
import scripts.summary_celltypes_IHOPE
import importlib

importlib.reload(scripts.summary_celltypes_IHOPE)

In [ ]:
from scripts.summary_celltypes_IHOPE import summarize_celltypes_IHOPE

df_summary = summarize_celltypes_IHOPE(
    adata,
    filename=f"{basename}_filtered_arcsinh_cf5.0_IHOPE_NEW_summary.csv" #New as in Apr 26
)
